# 35. MemoRAG

Memory-inspired RAG: compress the whole corpus into a global memory, then use it to draft retrieval clues.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

For very long inputs, the question often doesn't contain the right keywords: *'How did the main character's attitude change?'* MemoRAG gives the system a **global memory** - a compressed understanding of the whole corpus. Given a query, the memory drafts **clues** (likely evidence phrases), and *those* drive retrieval. Here the memory is a hierarchical LLM summary.

## How it works

1. Summarise groups of chunks, then merge into one compact memory
2. Given a query, ask the LLM (with the memory) for 3 clue sentences
3. Retrieve with the query **and** each clue
4. Merge the hits and generate

```text
 corpus --> compress --> GLOBAL MEMORY
 query + memory --> clues ("RRF uses 1/(60+rank)", ...) --> retrieve with clues --> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

MemoRAG uses a light long-context model as *global memory* that sees the whole database, produces **clues** (draft answers / evidence hints) for the query, and uses those clues to retrieve. Here the memory is a hierarchical LLM summary.

In [ ]:
# build the memory: summarise groups of chunks, then summarise the summaries
group_summaries = [llm("Summarise the key facts:\n\n" + "\n".join(chunks[i:i + 5])) for i in range(0, len(chunks), 5)]
memory = llm("Merge these notes into one compact memory of the whole corpus (max 250 words):\n\n" + "\n\n".join(group_summaries))
print(memory)

Given the query, the memory drafts clues - concrete phrases that the evidence is likely to contain.

In [ ]:
def memorag(query, k=3):
    clues = llm_json(
        f"Memory of the corpus:\n{memory}\n\nWrite 3 short clue sentences that the evidence for this question would likely contain. "
        f"Reply as a JSON list of strings.\n\nQuestion: {query}"
    ) or []
    seen, hits = set(), []
    for q in [query] + [c for c in clues if isinstance(c, str)]:
        for h in store.search(q, k):
            if h["idx"] not in seen:
                seen.add(h["idx"]); hits.append(h)
    return clues, hits[:k + 2]

query = "Which techniques help reduce the cost of a RAG system?"
clues, hits = memorag(query)
print("clues:", clues)
print(answer(query, [h["text"] for h in hits]))

## Clues vs the raw query

In [ ]:
for q in ["Which techniques help reduce the cost of a RAG system?", "How can I make answers more trustworthy?"]:
    clues, hits = memorag(q)
    print(f"Q: {q}\n   clues: {clues}\n   chunks retrieved: {[h['idx'] for h in hits]}")
    print("   plain top-3  :", [h['idx'] for h in store.search(q, 3)], "\n")

## Evaluate retrieval

In [ ]:
compare_retrievers({"plain query": store.search, "MemoRAG clues": lambda q, k=3: memorag(q, k)[1]}, k=3)

## Memory size vs corpus size

In [ ]:
print(f"corpus: {len(text.split())} words -> memory: {len(memory.split())} words ({len(memory.split()) / len(text.split()):.0%})")

## End-to-end comparison

In [ ]:
compare_pipelines({"plain RAG": lambda q: rag(q, store, 3)[0], "MemoRAG": lambda q: answer(q, [h["text"] for h in memorag(q)[1]])}, n=5)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Helps with vague or implicit queries | Memory build cost grows with the corpus |
| Global awareness of the corpus | The real MemoRAG uses a long-context memory model |
| Clues are inspectable | A lossy memory can mislead clues |

## When to use it

Tasks needing global understanding (long books, case files) and ambiguous questions.

## Experiments to try

- Refresh memory incrementally when documents are added.
- Increase memory size and plot hit rate.
- Add the clues to the final prompt.

## Key takeaways

- Memory -> clues -> retrieval.
- Good for vague questions.
- Memory quality caps performance.

---
**Next:** [36. Agentic RAG](../36_Agentic_RAG/Agentic_RAG.ipynb)